# 校准## 简介本教程介绍了如何使用 **scikit-rf** 校准从 VNA 获取的数据。有关 VNA 校准的介绍，请参阅 Rumiantsev 和 Ridler 的文章 [[1](#link_ref1)]，有关不同校准算法的概述，请参阅 Doug Rytting 的演示文稿 [[2](#link_ref2)]。如果您喜欢阅读书籍，那么您可以查阅 [[3](#link_ref3)]。以下是一些校准被测设备 (DUT) 的示例，假设您已经测量了一组可接受的校准标准，并获得了一组对应的理想响应。这可以被称为*离线*校准，因为它不是在 VNA 本身进行。这种技术的优点是，它为非传统校准提供了最大的灵活性，并保留了所有原始数据。scikit-rf 中实现了许多校准算法。选择取决于介质、可用的校准标准以及所需的准确性和工作量之间的权衡。scikit-rf 中可用的校准算法列在 [校准](../api/calibration/index.rst) API 页面中。

## 创建校准校准是通过一个 [校准](../api/calibration/index.rst) 类来执行的。一般来说，[校准](../api/calibration/index.rst) 对象需要两个参数：*   一个包含测量的 `Network` 对象的列表*   一个包含理想 `Network` 对象的列表两个列表中 `Network` 元素必须相似（端口数相同，频率信息等），并且必须对齐，这意味着理想列表中的第一个元素必须与测量列表中的第一个元素对应。自校准算法，例如 TRL，不需要预定义的理想响应。

## 单端口示例此示例代码旨在提供指导，而非追求简洁。为了构建一个单端口校准，您需要测量至少三个校准标准，并获得它们的已知*理想*响应，这些响应应以 `Network` 形式存储。这些 `Network` 可以从 Touchstone 文件中创建，所有现代 VNA 都支持这种格式。在下面的脚本中，用于传统短路/开路/负载/直通 (SOL) 校准套件的测量值和理想 Touchstone 文件分别位于 `measured/` 和 `ideal/` 文件夹中。这些文件用于创建一个 `OnePort` 校准，并校正测量的 DUT。```pythonimport skrf as rffrom skrf.calibration import OnePort## 创建 Calibration 类所需的数据# 一个 Network 类型的列表，用于存储“理想”响应my_ideals = [    rf.Network('ideal/short.s1p'),    rf.Network('ideal/open.s1p'),    rf.Network('ideal/load.s1p'),]# 一个 Network 类型的列表，用于存储“测量”响应my_measured = [    rf.Network('measured/short.s1p'),    rf.Network('measured/open.s1p'),    rf.Network('measured/load.s1p'),]## 创建一个 Calibration 实例cal = OnePort(    ideals=my_ideals,    measured=my_measured,)## 运行校准算法，并将其应用于 DUT# 运行校准算法cal.run()# 将其应用于 DUTdut = rf.Network('my_dut.s1p')dut_caled = cal.apply_cal(dut)dut_caled.name = dut.name + ' corrected'# 绘制结果dut_caled.plot_s_db()# 保存结果dut_caled.write_touchstone()```### 简洁的单端口示例此示例实现了与上述示例相同的任务，但采用了更简洁的*编程*方式。```pythonimport skrf as rffrom skrf.calibration import OnePortmy_ideals = rf.io.load_all_touchstones('ideals/')my_measured = rf.io.load_all_touchstones('measured/')## 创建一个 Calibration 实例cal = OnePort(    ideals=[my_ideals[k] for k in ['short', 'open', 'load']],    measured=[my_measured[k] for k in ['short', 'open', 'load']],)## 您对“cal”的操作可能与上述示例类似```

## 两端口校准显然，两端口校准比单端口校准更复杂。**skrf** 支持几种不同的两端口算法。传统的 `SOLT` 算法使用 12 项误差模型。此算法比较简单，类似于单端口示例。`EightTerm` 校准基于 R.A.Speciale 在 [[4](#link_ref4)] 中描述的算法。它可以由任意数量的校准标准构成，只要满足一些基本约束即可。简而言之，您需要三个两端口校准标准；其中一个必须是透过的，另一个必须提供已知的阻抗，并且是反射的。请注意，在文献中，术语 *8 项* 用于描述各种校准算法（如 TRL、LRM 等）使用的特定误差模型。`EightTerm` 类是上述算法的实现，它不使用任何自校准。使用 8 项误差模型公式的一个重要细节是，为了实现高质量的校准，可能需要测量开关项（感谢 Dylan Williams 指出这一点）。接下来将对此进行描述。### 开关项Roger Marks 最初在 [[5](#link_ref5)] 中描述了开关项，开关项考虑了 8 项（也称为 *误差盒*）模型过于简化这一事实。两个误差网络会根据激励哪个端口略有变化。这是由于 VNA 内部的开关造成的。因此，开关项描述了当源切换到另一个端口时的不理想负载。可以使用低插入损耗的透过标准（如直通）直接测量开关项，该透过标准连接到连接到仪器端口的电缆之间。有时，VNA 本身可以提供自定义的测量配置。**skrf** 支持在 `skrf.vi.vna` 模块中测量开关项，请参见 HP8510 的 `skrf.vi.vna.hp.HP8510C.get_switch_terms`，或 PNA 的 `skrf.vi.vna.keysight.get_switch_terms`。如果没有开关项测量，您的校准质量将根据 VNA 的特性而有所不同。### 在两端口校准中使用单端口理想值通常，您拥有理想反射标准的数据，这些数据以单端口触点文件的形式存在（即 `.s1p`）。要将其与 skrf 的两端口校准方法一起使用，您需要创建一个由两个网络组成的两个端口网络。函数 `skrf.network.two_port_reflect` 即可完成此操作。```pythonshort = rf.Network('ideals/short.s1p')shorts = rf.network.two_port_reflect(short, short)```### SOLT 示例两端口校准的执行方式与单端口校准相同，唯一的区别是所有校准标准都是两端口网络。这甚至适用于反射和负载标准（S21=S12=0）。因此，如果您测量反射或负载标准，则应同时测量其中两个，并将信息存储在两个端口网络中。例如，将一个短路连接到端口 1，将另一个短路连接到端口 2，并将两个端口的测量结果保存为“short,short.s2p”或类似的文件。或者，对于 `SOLT`、`TwelveTerm` 或 `EightTerm` 校准，也可以接受逐个端口进行测量，同时让另一个未使用的端口保持开路状态。稍后，可以使用 `skrf.network.two_port_reflect` 将两个单端口网络组合成一个两个端口网络，如上所述。默认情况下，不执行隔离校准。对于反射甚至负载标准，会忽略 $S_{21}$ 和 $S_{12}$。如果需要隔离校准，则必须明确告知 scikit-rf 执行此操作。具体来说，使用一个可选参数 `isolation` 来实现此目的。要校准隔离，请将此参数设置为一个两个端口网络，该网络表示在两个端口上都连接负载的测量结果。值得注意的是，不太常见的（但仍然受支持的）`SixteenTerm` 校准是值得注意的例外：此校准专门设计用于比标准的 `SOLT` 模型更好地消除端口泄漏。同时测量连接到两个端口的各种标准是*必需的*。```pythonimport skrf as rffrom skrf.calibration import SOLT# 一个列表，包含“理想”响应的 Network 类型my_ideals = [    rf.Network('ideal/short, short.s2p'),    rf.Network('ideal/open, open.s2p'),    rf.Network('ideal/load, load.s2p'),    rf.Network('ideal/thru.s2p'),]# 一个列表，包含“测量”响应的 Network 类型my_measured = [    rf.Network('measured/short, short.s2p'),    rf.Network('measured/open, open.s2p'),    rf.Network('measured/load, load.s2p'),    rf.Network('measured/thru.s2p'),]## 创建一个 SOLT 实例cal = SOLT(    ideals = my_ideals,    measured = my_measured,    rf.Network('measured/load, load.s2p'),    # 隔离校准是可选的，可以删除。)## 运行，并将校准应用于 DUTcal.run()## 将其应用于 DUTdut = rf.Network('my_dut.s2p')dut_caled = cal.apply_cal(dut)## 绘制结果dut_caled.plot_s_db()## 保存结果dut_caled.write_touchstone()```

## 保存和加载校准数据可以使用 pickles 的临时存储容器将[校准](../api/calibration/index.rst)数据写入磁盘并从磁盘读取。可以使用 `Calibration.write` 或 `rf.io.write()` 进行写入，使用 `rf.io.read()` 进行读取。由于这些函数依赖于序列化，因此不建议将其用于长期数据存储。目前，除了保存用于生成校准数据的脚本外，没有其他方法可以实现校准对象的长期存储。

## References<div id='link_ref1'>[1] [Andrej Rumiantsev and Nick Ridler, "VNA Calibration", IEEE Microwave Magazine 2008](http://anlage.umd.edu/Microwave%20Measurements%20for%20Personal%20Web%20Site/Rumiantsev%20VNA%20Calibration%20IEEE%20Mag%20June%202008%20page%2086.pdf)</div> <div id='link_ref2'>[2] [Doug Rytting, "Network Analyzer, Error Models and Calibration Methods"](http://emlab.uiuc.edu/ece451/appnotes/Rytting_NAModels.pdf)</div><div id='link_ref3'>[3] J. P. Dunsmore, "Handbook of microwave component measurements: with advanced vna techniques". Hoboken, N.J: Wiley, 2012.</div><div id='link_ref4'>[4] R.A. Speciale, "A Generalization of the TSD Network-Analyzer Calibration Procedure, Covering n-Port Scattering-Parameter Measurements, Affected by Leakage Errors," Microwave Theory and Techniques, IEEE Transactions on , vol.25, no.12, pp. 1100- 1115, Dec 1977. URL: http://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=1129282&isnumber=25047 </div><div id='link_ref5'>[5] Roger B. Marks, "Formulations of the Basic Vector Network Analyzer Error Model including Switch-Terms," ARFTG Conference Digest-Fall, 50th , vol.32, no., pp.115-126, Dec. 1997. URL: http://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=4119948&isnumber=4119931   </div>